# RNN Training - Optimized Version
## Key Improvements:
1. **Clean imports** - unified to `tensorflow.keras`
2. **EarlyStopping & ReduceLROnPlateau** - prevent overfitting, auto-reduce LR
3. **BatchNormalization** - stabilize training
4. **tf.data pipeline** - accelerate data loading
5. **Mixed Precision** - speed up GPU training
6. **Fixed data leakage** - scaler fitted on train data only
7. **Replaced `accuracy` metric** with proper R2 score (accuracy is meaningless for regression)
8. **Changed activation** - `tanh` instead of `relu` (standard for RNN)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import timeit
import os

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, SimpleRNN
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, CSVLogger, ModelCheckpoint
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Enable Mixed Precision for faster GPU training (if available)
tf.keras.mixed_precision.set_global_policy('mixed_float16')

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU available: {tf.config.list_physical_devices("GPU")}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Hyperparameter Configuration

In [ ]:
# ============================================================
# HYPERPARAMETERS - Adjust all settings here
# ============================================================
CONFIG = {
    # Data
    'data_path': '/content/drive/MyDrive/Research/Project Man/Data1/Out_ML_cm.csv',
    'n_features_in': 5,        # Number of input columns (0:5)
    'n_features_out': 200,     # Number of output columns (6:206)
    'train_ratio': 0.8,
    'n_samples': 30000,
    
    # Model architecture
    'rnn_units': [128, 128, 64, 64],   # Number of units per RNN layer
    'dense_units': [32],               # Dense layer units before output
    'dropout_rate': 0.25,
    
    # Training
    'epochs': 1000,
    'batch_size': 128,
    'learning_rate': 0.001,
    'patience_es': 50,
    'patience_lr': 20,
    'min_lr': 1e-6,
    
    # Output
    'output_dir': '/content/drive/MyDrive/Research/Project Man/RNN/',
    'model_name': 'RNNcm',
}

os.makedirs(CONFIG['output_dir'], exist_ok=True)

## 2. Load & Preprocess Data

In [ ]:
start = timeit.default_timer()

# Load data from CSV
dataframe = pd.read_csv(CONFIG['data_path'], header=None)
dataset = dataframe.values[:CONFIG['n_samples'], :206]
print(f'Dataset shape: {dataset.shape}')

# Use separate scalers for input and output
# This allows inverse_transform on predictions later
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

n_in = CONFIG['n_features_in']
n_split = int(CONFIG['n_samples'] * CONFIG['train_ratio'])

X = dataset[:, :n_in]
y = dataset[:, 6:206]

# FIX: Fit scaler on training data ONLY to prevent data leakage
# (Original code fitted on the entire dataset including test data,
#  which means test set statistics leak into the training process)
scaler_X.fit(X[:n_split])
scaler_y.fit(y[:n_split])

X = scaler_X.transform(X)
y = scaler_y.transform(y)

# Train/Test split
x_train, x_test = X[:n_split], X[n_split:]
y_train, y_test = y[:n_split], y[n_split:]

# Reshape for model input: (samples, 1, features)
x_train = x_train.reshape(x_train.shape[0], 1, x_train.shape[1])
x_test = x_test.reshape(x_test.shape[0], 1, x_test.shape[1])

input_node = x_train.shape[2]
output_node = y_train.shape[1]

print(f'x_train: {x_train.shape}, y_train: {y_train.shape}')
print(f'x_test:  {x_test.shape},  y_test:  {y_test.shape}')

## 3. Create tf.data Pipeline (faster data loading)

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
batch_size = CONFIG['batch_size']

# Build optimized training dataset:
# - shuffle: randomize sample order each epoch
# - batch: group samples into batches
# - prefetch: prepare next batch while GPU processes current one
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(buffer_size=len(x_train))
    .batch(batch_size)
    .prefetch(AUTOTUNE)
)

# Validation dataset (no shuffle needed for evaluation)
val_ds = (
    tf.data.Dataset.from_tensor_slices((x_test, y_test))
    .batch(batch_size)
    .prefetch(AUTOTUNE)
)

## 4. Build Model
**Improvements over original:**
- Added `BatchNormalization` between RNN layers for stable gradients
- Changed activation from `relu` to `tanh` (standard for SimpleRNN)
- Added `Dropout` for regularization
- Output layer uses `float32` (required for mixed precision)

In [ ]:
def build_model(config):
    """Build SimpleRNN model from config dictionary."""
    units = config['rnn_units']
    dense_units = config['dense_units']
    input_shape = (1, config['n_features_in'])
    
    model = Sequential(name='RNN_optimized')
    
    # RNN Layer 1 + BatchNorm
    model.add(SimpleRNN(units[0], return_sequences=True, activation='tanh',
                        input_shape=input_shape))
    model.add(BatchNormalization())
    
    # RNN Layer 2 + BatchNorm
    model.add(SimpleRNN(units[1], return_sequences=True, activation='tanh'))
    model.add(BatchNormalization())
    
    # RNN Layer 3 + BatchNorm
    model.add(SimpleRNN(units[2], return_sequences=True, activation='tanh'))
    model.add(BatchNormalization())
    
    # RNN Layer 4 (final, no return_sequences) + BatchNorm + Dropout
    model.add(SimpleRNN(units[3], activation='tanh'))
    model.add(BatchNormalization())
    model.add(Dropout(config['dropout_rate']))
    
    # Dense hidden layer
    for u in dense_units:
        model.add(Dense(u, activation='relu'))
    
    # Output layer - dtype must be float32 when using mixed precision
    model.add(Dense(config['n_features_out'], dtype='float32'))
    
    return model

model = build_model(CONFIG)
model.summary()

## 5. Custom R2 Metric

In [ ]:
# Custom R2 metric compatible with mixed precision training
#
# FIX: The original r2_score function computed mean(y_true) per batch,
# which gives incorrect R2 values because the global mean differs from
# batch-level means. This class accumulates sum, sum-of-squares, and
# count across all batches, then computes the correct R2 at epoch end.

class R2Score(tf.keras.metrics.Metric):
    def __init__(self, name='r2_score', **kwargs):
        super().__init__(name=name, **kwargs)
        self.ss_res = self.add_weight(name='ss_res', initializer='zeros')
        self.y_sum = self.add_weight(name='y_sum', initializer='zeros')
        self.y_sq_sum = self.add_weight(name='y_sq_sum', initializer='zeros')
        self.n = self.add_weight(name='n', initializer='zeros')
    
    def update_state(self, y_true, y_pred, sample_weight=None):
        # Cast to float32 for numerical stability with mixed precision
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        # Accumulate residual sum of squares
        self.ss_res.assign_add(tf.reduce_sum(tf.square(y_true - y_pred)))
        # Accumulate statistics for total sum of squares
        self.y_sum.assign_add(tf.reduce_sum(y_true))
        self.y_sq_sum.assign_add(tf.reduce_sum(tf.square(y_true)))
        self.n.assign_add(tf.cast(tf.size(y_true), tf.float32))
    
    def result(self):
        # R2 = 1 - SS_res / SS_tot
        # SS_tot = sum(y^2) - n * mean(y)^2 (computed from accumulated stats)
        y_mean = self.y_sum / (self.n + tf.keras.backend.epsilon())
        ss_tot = self.y_sq_sum - self.n * tf.square(y_mean)
        return 1.0 - self.ss_res / (ss_tot + tf.keras.backend.epsilon())
    
    def reset_state(self):
        # Reset all accumulators at the start of each epoch
        self.ss_res.assign(0.0)
        self.y_sum.assign(0.0)
        self.y_sq_sum.assign(0.0)
        self.n.assign(0.0)

## 6. Compile & Train

In [ ]:
# Define callbacks for training control
callbacks = [
    # Stop training early if validation loss doesn't improve for patience epochs
    EarlyStopping(
        monitor='val_loss',
        patience=CONFIG['patience_es'],
        restore_best_weights=True,  # Restore weights from the best epoch
        verbose=1
    ),
    # Halve learning rate when validation loss plateaus
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,                 # Multiply LR by 0.5
        patience=CONFIG['patience_lr'],
        min_lr=CONFIG['min_lr'],    # Don't reduce below this value
        verbose=1
    ),
    # Save the best model checkpoint based on lowest validation loss
    ModelCheckpoint(
        filepath=os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_best.keras'),
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    # Log training history to CSV file for later analysis
    CSVLogger(os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_history.csv')),
]

# Compile model with Adam optimizer and MSE loss
optimizer = Adam(learning_rate=CONFIG['learning_rate'])
model.compile(
    loss='mse',
    optimizer=optimizer,
    metrics=[R2Score()]
)

# Train the model using tf.data pipeline
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=CONFIG['epochs'],
    callbacks=callbacks,
    verbose=1
)

## 7. Evaluate Model

In [ ]:
# Evaluate on validation set
scores = model.evaluate(val_ds, verbose=1)

# Generate predictions for both train and test sets
pred_train = model.predict(x_train, batch_size=CONFIG['batch_size'])
pred_test = model.predict(x_test, batch_size=CONFIG['batch_size'])

# Compute metrics using sklearn (more accurate than batch-wise Keras metrics)
metrics = {
    'mse_train': mean_squared_error(y_train, pred_train),
    'mse_test': mean_squared_error(y_test, pred_test),
    'mae_train': mean_absolute_error(y_train, pred_train),
    'mae_test': mean_absolute_error(y_test, pred_test),
    'r2_train': r2_score(y_train, pred_train),
    'r2_test': r2_score(y_test, pred_test),
}

# Record total computing time
stop = timeit.default_timer()
metrics['computing_time'] = stop - start

# Print all metrics
for k, v in metrics.items():
    print(f'{k}: {v}')
print(f'\nModel scores: {scores}')

## 8. Save Results

In [ ]:
# Save complete model (architecture + weights + optimizer state)
model.save(os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_final.keras'))

# Save model architecture as JSON (for loading structure without weights)
json_string = model.to_json()
with open(os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '.json'), 'w') as f:
    f.write(json_string)

# Save all evaluation metrics to a text file
output_txt = os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_metrics.txt')
with open(output_txt, 'w') as f:
    for k, v in metrics.items():
        f.write(f'{k}: {v}\n')
    f.write(f'scores: {scores}\n')
    f.write(f'{model.metrics_names[1]}: {scores[1]*100:.2f}%\n')

print('All results saved successfully.')

## 9. Visualize Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Training & Validation Loss (log scale for better visibility)
axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Loss (MSE)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_yscale('log')

# Plot 2: Training & Validation R2 Score
axes[1].plot(history.history['r2_score'], label='Train')
axes[1].plot(history.history['val_r2_score'], label='Validation')
axes[1].set_title('R² Score')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('R²')
axes[1].legend()

# Plot 3: Learning Rate schedule (recorded by ReduceLROnPlateau callback)
if 'lr' in history.history:
    axes[2].plot(history.history['lr'])
    axes[2].set_title('Learning Rate')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('LR')
    axes[2].set_yscale('log')
else:
    axes[2].text(0.5, 0.5, 'LR not recorded', 
                 ha='center', va='center', transform=axes[2].transAxes)

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_curves.png'), dpi=150)
plt.show()

## 10. Prediction vs Ground Truth

In [ ]:
# Randomly select test samples to visualize prediction quality
n_show = 5
fig, axes = plt.subplots(n_show, 1, figsize=(14, 3 * n_show))

for i in range(n_show):
    idx = np.random.randint(0, len(x_test))
    axes[i].plot(y_test[idx], label='Ground Truth', alpha=0.8)
    axes[i].plot(pred_test[idx], label='Prediction', alpha=0.8)
    axes[i].set_title(f'Test Sample #{idx}')
    axes[i].legend()

plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_dir'], CONFIG['model_name'] + '_predictions.png'), dpi=150)
plt.show()